<a href="https://colab.research.google.com/github/Lornatt88/maai902-nlp-solution/blob/main/notebooks/02b_long_sentences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2b: Shortening long sentences
Some sentences are longer than BERT's 128-token limit. Instead of cutting off their ending, an LLM picks the most important parts, so the meaning is kept.

## 1. Counting long sentences
Counts how many sentences in each split are longer than 128 tokens, using Legal-BERT's tokenizer (the model we will train).

In [1]:
!pip install -q datasets transformers

from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset("coastalcph/lex_glue", "unfair_tos")
tok = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")

for split in ["train", "validation", "test"]:
    lengths = [len(tok(t)["input_ids"]) for t in ds[split]["text"]]      # length of every sentence
    long = sum(1 for n in lengths if n > 128)                              # how many are too long
    print(f"{split}: {long} sentences longer than 128 tokens (longest: {max(lengths)})")

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

unfair_tos/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  501kB            

unfair_tos/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

unfair_tos/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  147kB            

unfair_tos/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

train: 71 sentences longer than 128 tokens (longest: 478)
validation: 39 sentences longer than 128 tokens (longest: 288)
test: 18 sentences longer than 128 tokens (longest: 331)


Using Legal-BERT’s tokenizer, 71 training, 39 validation and 18 test sentences are longer than 128 tokens (128 in total, about 1.4% of the data). These are the sentences that will be shortened.

## 2. Collecting the long sentences
Gathers every sentence longer than 128 tokens and remembers which split and position it came from, so the shortened version can be put back in the right place.

In [2]:
long_items = []    # (split, position, text) for every long sentence

for split in ["train", "validation", "test"]:
    for i, t in enumerate(ds[split]["text"]):
        if len(tok(t)["input_ids"]) > 128:
            long_items.append((split, i, t))

print("Long sentences collected:", len(long_items))
print("\nExample:", long_items[0][2])

Long sentences collected: 128

Example: to the fullest extent permitted by applicable law , in no event will tinder , its affiliates , employees , licensors or service providers be liable for any indirect , consequential , exemplary , incidental , special or punitive damages , including , without limitation , loss of profits , whether incurred directly or indirectly , or any loss of data , use , goodwill , or other intangible losses , resulting from : ( i ) your access to or use of or inability to access or use the services , ( ii ) the conduct or content of other users or third parties on , through , or following use of the services ; or ( iii ) unauthorized access , use or alteration of your content , even if tinder has been advised of the possibility of such damages . 



## 3. Loading the LLM
Loads Qwen2.5-7B-Instruct, a free open model, compressed to 4-bit so it fits on Colab's free GPU. A quick test checks that it answers.

In [3]:
!pip install -q bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

llm_name = "Qwen/Qwen2.5-7B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto",
)

def ask_llm(prompt, max_new_tokens=200):
    """Send one message to the LLM and return its answer."""
    messages = [{"role": "user", "content": prompt}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)   # no randomness
    return llm_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask_llm("Say hello in one short sentence."))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.7 MB/s eta 0:00:00


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Hello!


## 4. Extractive summaries
Asks the LLM for an extractive summary of each long sentence: the most important exact phrases, copied word for word. Each phrase is checked to make sure it appears exactly in the original, and the phrases are joined in their original order with "...". If the result is longer than 128 tokens, the LLM first picks the phrase with the main rule (who does or promises what), which is always kept. Then it removes the least important of the other phrases, one at a time, until the summary fits.

Quote marks are converted to the dataset's style before checking, and if the LLM returns the whole sentence as one long phrase, it is asked to split that phrase into smaller exact phrases.

If the LLM leaves out a small part inside a phrase (like a list marker), the phrase is split into exact pieces, but only if every word it wrote appears in the original in the same order.

In [17]:
import re

PROMPT = """Make an extractive summary of this clause from an online Terms of Service.

List the most important phrases from the clause, one per line.
Rules:
- Copy each phrase exactly as it appears in the clause. Never change, reword or add words.
- Choose the phrases that say what the company or the user can or cannot do: rights, obligations, limits on liability, changes to the terms, account termination, content removal, how disputes are handled, and which laws or courts apply.
- Skip examples, long lists and background.
- All phrases together must be at most 70 words.
- Return only the phrases, one per line, without numbers or quotation marks.

Example:
Clause: we may , at our sole discretion and for any reason , including but not limited to violations of these terms or suspected fraud , suspend or terminate your account at any time without notice .
Phrases:
we may , at our sole discretion and for any reason
suspend or terminate your account at any time without notice

Clause: {clause}
Phrases:"""

MAIN_PROMPT = """These phrases come from a clause in an online Terms of Service.
Which ONE phrase states the main rule of the clause (who does or promises what)?
Copy that phrase exactly and return only it.

Phrases:
{phrases}"""

DROP_PROMPT = """These phrases come from a clause in an online Terms of Service.
Which ONE phrase is the least important for understanding what the company or the user can or cannot do?
Copy that phrase exactly and return only it.

Phrases:
{phrases}"""

def words(s):
    """Just the words, lowercase, without punctuation."""
    return re.findall(r"[a-z0-9]+", s.lower())

def clean_line(line):
    """Remove list markers, and quote marks only if they wrap the whole line."""
    l = line.strip(" -*•")
    if len(l) > 1 and l[0] in '"“' and l[-1] in '"”':
        l = l[1:-1]
    return l

def to_dataset_style(s):
    """Format like the dataset: lowercase, its quote style (`` ''), spaces around punctuation, "'s" split off."""
    s = s.lower().strip()
    s = s.replace("“", '"').replace("”", '"').replace("’", "'")
    s = re.sub(r'"([^"]*)"', r" `` \1 '' ", s)          # "word" -> `` word ''
    s = re.sub(r"(\w)'s\b", r"\1 's", s)
    s = re.sub(r"\s*([,;:()])\s*", r" \1 ", s)
    s = re.sub(r"\s*\.$", " .", s)
    return re.sub(r"\s+", " ", s).strip()

def join(found):
    """Join phrases (already in original order) with '...'."""
    return " ... ".join(p for _, p in found)

def exact_pieces(phrase, original):
    """Split a phrase into pieces that each appear exactly in the original, in order.
    Every word of the phrase must be found; if any word was added or changed, return None."""
    ws = phrase.split()
    padded = " " + original + " "
    pieces, start, pos = [], 0, 0
    while start < len(ws):
        for k in range(len(ws), start, -1):                  # try the longest piece first
            cand = " " + " ".join(ws[start:k]) + " "
            p = padded.find(cand, pos)
            if p != -1:
                pieces.append((p, " ".join(ws[start:k])))
                pos = p + len(cand) - 1                       # next piece must come after this one
                start = k
                break
        else:
            return None                                       # a word was added or changed: reject
    return pieces

def get_phrases(clause, original):
    """Ask the LLM for phrases; keep every phrase whose words all appear in the original, in order."""
    answer = ask_llm(PROMPT.format(clause=clause), max_new_tokens=250)
    lines = [l for l in answer.splitlines() if l.strip()]
    found = []
    for line in lines:
        phrase = to_dataset_style(clean_line(line))
        pieces = exact_pieces(phrase, original)
        if pieces and len(words(phrase)) >= 2:
            for item in pieces:
                if item[1] not in [p for _, p in found]:
                    found.append(item)
    found.sort()                                              # original order
    return found, len(lines)

def pick(prompt, options):
    """Ask the LLM to pick one phrase from the options. Returns the matching option, or None."""
    listing = "\n".join(p for _, p in options)
    answer = to_dataset_style(clean_line(ask_llm(prompt.format(phrases=listing), max_new_tokens=80)))
    for item in options:
        if item[1] == answer:
            return item
    return None

def too_long(found):
    return len(tok(join(found))["input_ids"]) > 128

def shorten(text):
    """Exact phrases, split a giant one if needed, lock the main rule, drop the least important until it fits."""
    found, n_lines = get_phrases(text, text)
    notes = []
    if len(found) == 1 and too_long(found):                  # one giant phrase: ask to split it
        sub, _ = get_phrases(found[0][1], text)
        if len(sub) > 1:
            found = sub
            notes.append("split giant phrase")
    if not found:
        return None, "failed"

    if too_long(found):
        main = pick(MAIN_PROMPT, found)                      # lock the main rule
        notes.append("main: llm" if main else "main: first phrase")
        main = main or found[0]
        while too_long(found):
            others = [x for x in found if x != main]
            if not others:
                return None, "failed"
            drop = pick(DROP_PROMPT, others)
            notes.append("drop: llm" if drop else "drop: longest")
            drop = drop or max(others, key=lambda x: len(x[1]))
            found = [x for x in found if x != drop]

    if len(words(join(found))) < 5:
        return None, "failed"
    return join(found), f"kept {len(found)} of {n_lines} phrases | {notes}"

# Test on the first 3 long sentences
for split, i, t in long_items[:3]:
    short, method = shorten(t)
    print(f"{split} #{i} | {method}")
    if short:
        print(f"Tokens: {len(tok(t)['input_ids'])} -> {len(tok(short)['input_ids'])}")
        print("SUMMARY:", short)
    print()

train #164 | kept 7 of 8 phrases | ['main: llm', 'drop: llm']
Tokens: 153 -> 107
SUMMARY: to the fullest extent permitted by applicable law ... in no event will tinder , its affiliates , employees , licensors or service providers be liable ... indirect , consequential , exemplary , incidental , special or punitive damages ... loss of profits , whether incurred directly or indirectly ... loss of data , use , goodwill , or other intangible losses ... your access to or use of or inability to access or use the services ... unauthorized access , use or alteration of your content

train #234 | kept 5 of 6 phrases | ['main: llm', 'drop: llm', 'drop: llm']
Tokens: 145 -> 119
SUMMARY: we can not ensure the security or privacy of information you provide through the internet and your email messages ... you release us from any and all liability in connection with the use of such information by other parties ... we are not responsible for , and can not control , the use by others of any information

## 4a. Checking why summaries fail
Shows the LLM's raw answer for one sentence at each word limit, and which check fails, to find what needs fixing.

In [6]:
def show_attempt(text, limit=70):
    """Show each phrase the LLM gives, whether it matches the original exactly, and the total length."""
    answer = ask_llm(PROMPT.format(limit=limit, clause=text), max_new_tokens=250)
    found = []
    for line in answer.splitlines():
        if not line.strip():
            continue
        phrase = to_dataset_style(line.strip(' "-*•'))
        ok = text.find(phrase) != -1
        print("✓" if ok else "✗", phrase)
        if ok:
            found.append(phrase)
    print("Tokens of joined summary:", len(tok(" ... ".join(found))["input_ids"]), "\n")

for idx in [0, 2]:          # the two that failed
    print(f"===== {long_items[idx][0]} #{long_items[idx][1]} =====")
    show_attempt(long_items[idx][2])

===== train #164 =====
✓ to the fullest extent permitted by applicable law
✓ in no event will tinder , its affiliates , employees , licensors or service providers be liable
✓ indirect , consequential , exemplary , incidental , special or punitive damages
✓ loss of profits , whether incurred directly or indirectly
✓ loss of data , use , goodwill , or other intangible losses
✓ your access to or use of or inability to access or use the services
✓ conduct or content of other users or third parties on , through , or following use of the services
✓ unauthorized access , use or alteration of your content
Tokens of joined summary: 130 

===== train #493 =====
✓ represents , warrants and agrees that it will not contribute any subscriber content that
✓ reveals any trade secret , unless subscriber owns the trade secret or has the owner 's permission to post it
✓ infringes any intellectual property right of another or the privacy or publicity rights of another
✓ is libelous , defamatory , abusive 

## 4b. Shortening all long sentences
Runs the extractive summary on all 128 long sentences, counts how many succeeded and who chose each dropped phrase, and saves the results to Google Drive.

In [19]:
from tqdm.auto import tqdm
import pandas as pd
import os
from google.colab import drive

results = []
for split, i, t in tqdm(long_items):                     # progress bar
    short, method = shorten(t)
    results.append({
        "split": split, "index": i, "original": t, "summary": short, "method": method,
        "tokens_before": len(tok(t)["input_ids"]),
        "tokens_after": len(tok(short)["input_ids"]) if short else None,
    })

df = pd.DataFrame(results)
print("Shortened:", df["summary"].notna().sum(), "of", len(df))
print("Failed:", df["summary"].isna().sum())
print("Choices made by the LLM:", df["method"].str.count("llm").sum())
print("Choices made by a backup:", df["method"].str.count("first phrase|longest").sum())

# Save to Drive so the training notebooks can use the summaries
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/fine_print_detector", exist_ok=True)
df.to_csv("/content/drive/MyDrive/fine_print_detector/long_sentence_summaries.csv", index=False)
print("Saved to Drive.")

  0%|          | 0/128 [00:00<?, ?it/s]

Shortened: 126 of 128
Failed: 2
Choices made by the LLM: 166
Choices made by a backup: 47
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Drive.


## 4c. Saving the summaries
Saves the results table to Google Drive (and a copy in Colab), so the training notebooks can load it without running the LLM again.

In [11]:
from google.colab import drive

df.to_csv("long_sentence_summaries.csv", index=False)       # quick local copy first, just in case
drive.mount("/content/drive", force_remount=True)           # try connecting to Drive again
os.makedirs("/content/drive/MyDrive/fine_print_detector", exist_ok=True)
df.to_csv("/content/drive/MyDrive/fine_print_detector/long_sentence_summaries.csv", index=False)
print("Saved to Drive.")

Mounted at /content/drive
Saved to Drive.


## 4d. Looking at failures and backups
Counts which backups were used, shows which splits the failed sentences are in, and shows the LLM's phrases for two of the failed sentences.

In [12]:
print("Main phrase chosen by backup (first phrase):", df["method"].str.count("main: first phrase").sum())
print("Drops chosen by backup (longest):", df["method"].str.count("drop: longest").sum())

failed = df[df["summary"].isna()]
print("\nFailed per split:")
print(failed["split"].value_counts())

for _, row in failed.head(2).iterrows():
    print(f"\n===== {row['split']} #{row['index']} ({row['tokens_before']} tokens) =====")
    show_attempt(row["original"])

Main phrase chosen by backup (first phrase): 12
Drops chosen by backup (longest): 19

Failed per split:
split
train         7
test          3
validation    2
Name: count, dtype: int64

===== train #1709 (134 tokens) =====
✓ if you purchased `` game time '' under a program that provided you the right to utilize the service for a set period of time at a discounted rate ( `` discounted fees '' ) , and you request that blizzard entertainment terminate your account , blizzard entertainment shall provide a pro rata refund calculated by deducting from the discounted fees an amount equal to the monthly fee that is in effect at as of the date you request termination of your account divided by 30 , and then multiply that amount times with the number of days that you have utilized the service in the billing period that you purchased for the discounted fees
Tokens of joined summary: 133 


===== train #3310 (135 tokens) =====
✗ by "facebook" or "facebook services" we mean the features and services

## 4e. Retrying the failed sentences
Runs the improved summary step again on only the sentences that failed, updates the results table, and saves it to Drive again.

In [18]:
failed_rows = df[df["summary"].isna()].index

for k in tqdm(failed_rows):
    short, method = shorten(df.at[k, "original"])
    df.at[k, "summary"] = short
    df.at[k, "method"] = method
    df.at[k, "tokens_after"] = len(tok(short)["input_ids"]) if short else None

print("Shortened:", df["summary"].notna().sum(), "of", len(df))
print("Still failed:", df["summary"].isna().sum())
print("Choices made by the LLM:", df["method"].str.count("llm").sum())
print("Choices made by a backup:", df["method"].str.count("first phrase|longest").sum())
print("Giant phrases split:", df["method"].str.count("split giant phrase").sum())

df.to_csv("/content/drive/MyDrive/fine_print_detector/long_sentence_summaries.csv", index=False)
print("Saved to Drive.")

  0%|          | 0/9 [00:00<?, ?it/s]

Shortened: 126 of 128
Still failed: 2
Choices made by the LLM: 123
Choices made by a backup: 34
Giant phrases split: 1
Saved to Drive.


## 4f. Why the remaining sentences fail
For each sentence that still has no summary, shows how many phrases the LLM gave, how many matched the original exactly, and how long the matched phrases are together.

In [15]:
for _, row in df[df["summary"].isna()].iterrows():
    found, n = get_phrases(row["original"], row["original"])
    joined_tokens = len(tok(join(found))["input_ids"]) if found else 0
    print(f"{row['split']} #{row['index']} | {row['tokens_before']} tokens | "
          f"LLM gave {n} phrases, {len(found)} matched exactly | matched phrases together: {joined_tokens} tokens")

train #3310 | 135 tokens | LLM gave 1 phrases, 0 matched exactly | matched phrases together: 0 tokens
train #3404 | 200 tokens | LLM gave 7 phrases, 0 matched exactly | matched phrases together: 0 tokens
train #3692 | 283 tokens | LLM gave 5 phrases, 0 matched exactly | matched phrases together: 0 tokens
train #4791 | 246 tokens | LLM gave 28 phrases, 0 matched exactly | matched phrases together: 0 tokens
train #5351 | 218 tokens | LLM gave 27 phrases, 0 matched exactly | matched phrases together: 0 tokens
validation #1406 | 214 tokens | LLM gave 5 phrases, 0 matched exactly | matched phrases together: 0 tokens
validation #2239 | 188 tokens | LLM gave 4 phrases, 0 matched exactly | matched phrases together: 0 tokens
test #142 | 218 tokens | LLM gave 1 phrases, 1 matched exactly | matched phrases together: 218 tokens
test #1013 | 193 tokens | LLM gave 6 phrases, 0 matched exactly | matched phrases together: 0 tokens


## 4g. Finding where phrases stop matching
For two failed sentences, shows each phrase the LLM gave and the exact point where it stops matching the original, to find the formatting difference.

In [16]:
def where_it_breaks(phrase, original):
    """Return the longest start of the phrase found in the original, and the text right after it."""
    for k in range(len(phrase), 0, -1):
        if phrase[:k] in original:
            return phrase[:k][-40:], phrase[k:k + 30]
    return "", phrase[:30]

for split, idx in [("train", 3404), ("train", 3692)]:
    original = df[(df["split"] == split) & (df["index"] == idx)]["original"].iloc[0]
    answer = ask_llm(PROMPT.format(clause=original), max_new_tokens=250)
    print(f"===== {split} #{idx} =====")
    for line in [l for l in answer.splitlines() if l.strip()][:3]:
        phrase = to_dataset_style(clean_line(line))
        ok_part, breaks_at = where_it_breaks(phrase, original)
        print(f"...{ok_part} ❌ breaks at: {breaks_at!r}")
    print()

===== train #3404 =====
...you shall not  ❌ breaks at: 'take any action that imposes o'
...you shall not  ❌ breaks at: 'interfere or attempt to interf'
...you shall not  ❌ breaks at: 'bypass any measures vivino may'

===== train #3692 =====
...may also allow you to  ❌ breaks at: 'create 15-second videos that i'
...may also allow you to  ❌ breaks at: 'save your user videos locally '
...may also allow you to  ❌ breaks at: 'upload your user videos to cer'



## 4h. Checking the saved file
Reads the saved file back from Google Drive to confirm all rows and summaries are there.

In [20]:
import time

path = "/content/drive/MyDrive/fine_print_detector/long_sentence_summaries.csv"
check = pd.read_csv(path)

print("Rows in file:", len(check))
print("Rows with a summary:", check["summary"].notna().sum())
print("Last saved:", time.ctime(os.path.getmtime(path)))

drive.flush_and_unmount()          # make sure everything is fully synced to Drive
print("Drive synced.")

Rows in file: 128
Rows with a summary: 126
Last saved: Sat Oct 17 19:43:35 2026
Drive synced.


## 5. Loading the saved summaries
Loads the summaries saved in Google Drive, so the LLM does not need to run again, and lists the sentences that still have no summary.

In [2]:
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")
df = pd.read_csv("/content/drive/MyDrive/fine_print_detector/long_sentence_summaries.csv")

print("Rows:", len(df), "| With a summary:", df["summary"].notna().sum())
print("\nStill failed:")
print(df[df["summary"].isna()][["split", "index", "tokens_before"]])

Mounted at /content/drive
Rows: 128 | With a summary: 126

Still failed:
     split  index  tokens_before
39   train   3310            135
112   test    142            218


## 6. The two remaining failures
Prints the two sentences that could not be shortened, to decide how to handle them.

In [3]:
for _, row in df[df["summary"].isna()].iterrows():
    print(f"===== {row['split']} #{row['index']} ({row['tokens_before']} tokens) =====")
    print(row["original"], "\n")

===== train #3310 (135 tokens) =====
by `` facebook '' or '' facebook services '' we mean the features and services we make available , including through ( a ) our website at www.facebook.com and any other facebook branded or co-branded websites ( including sub-domains , international versions , widgets , and mobile versions ) ; ( b ) our platform ; ( c ) social plugins such as the like button , the share button and other similar offerings ; and ( d ) other media , brands , products , services , software ( such as a toolbar ) , devices , or networks now existing or later developed . 
 

===== test #142 (218 tokens) =====
neither academia.edu nor any other person or entity involved in creating , producing , or delivering the site , services or collective content will be liable for any incidental , special , exemplary or consequential damages , or lost profits , loss of data or loss of goodwill , service interruption , computer damage or system failure or the cost of substitute products 

In [4]:
print("train #3310 labels:", ds["train"]["labels"][3310])
print("test #142 labels:", ds["test"]["labels"][142])

train #3310 labels: []
test #142 labels: [0]


## 7. Hand check
Shows 10 random summaries with their unfair type, to check by reading that each summary still contains the part that makes the clause (un)fair. Scores: +1 key part kept and clear, 0 key part kept but awkward, -1 key part lost.

In [5]:
names = ds["train"].features["labels"].feature.names
sample = df[df["summary"].notna()].sample(10, random_state=7)     # same 10 every time

for n, (_, row) in enumerate(sample.iterrows(), 1):
    labs = ds[row["split"]]["labels"][int(row["index"])]
    types = [names[l] for l in labs] or "fair"
    print(f"#{n} | {row['split']} #{row['index']} | {row['tokens_before']} -> {int(row['tokens_after'])} tokens | type: {types}")
    print("SUMMARY:", row["summary"], "\n")

#1 | test #156 | 153 -> 122 tokens | type: ['Arbitration']
SUMMARY: you and academia.edu agree that any dispute , claim or controversy arising out of or relating to these terms or the breach , termination , enforcement , interpretation or validity thereof or the use of the site or services ( collectively , `` disputes '' ) will be settled by binding arbitration ... to seek injunctive or other equitable relief in a court of competent jurisdiction to prevent the actual or threatened infringement , misappropriation or violation of a party 's copyrights , trademarks , trade secrets , patents or other intellectual property rights ( ... an `` ip protection action '' ) 

#2 | validation #1962 | 239 -> 117 tokens | type: fair
SUMMARY: you represent and warrant that you are at least the age of majority in the state in which you reside and are the parent or legal guardian ... or have all proper consents from the parent or legal guardian ... of any minor who is depicted in or contributed to any u

In a manual check of 10 random summaries, 9 kept the key part of the clause (6 clear, 3 awkward but complete) and 1 lost its main rule. Both unfair clauses in the sample (Arbitration and Limitation of liability) kept the part that makes them unfair.